# Exploratory Data Analysis Workshop

Welcome! In this workshop you will practice the core steps of **Exploratory Data Analysis (EDA)** — the essential first phase of any data science project. Before you model, predict, or conclude anything, you need to *understand your data*.

### What is EDA?

EDA is a systematic approach to investigating a dataset. It was popularized by statistician **John Tukey** (1977), who argued that analysts should first explore data with an open mind before testing hypotheses. The goals are:

1. **Understand structure** — What variables exist? What types are they? How big is the data?
2. **Assess quality** — Are there missing values, duplicates, inconsistencies, or outliers?
3. **Discover patterns** — What distributions, correlations, and group differences exist?
4. **Generate questions** — What follow-up analyses or data collection might be needed?

### Choose your dataset

**Option A (recommended):** Bring a dataset related to your own challenge or project. Any tabular CSV will work.

**Option B:** Use the provided bike-sharing dataset (`data/bike_sharing_eda.csv`) — ~800 records from a fictional city bike-sharing system. The data is intentionally *messy*, just like real-world data.

### How this workshop works

- Exercises are marked with difficulty: ⭐ (basic), ⭐⭐ (intermediate), ⭐⭐⭐ (challenge)
- Everyone should complete the ⭐ exercises
- Move at your own pace — it's fine to skip ahead if you're comfortable
- **Discuss with your neighbors** — EDA is often a collaborative activity
- Exercises are written generically — adapt them to your dataset
- 💡 hints for the bike-sharing dataset are included in some exercises

---

---
## Part 0: Git Setup (~5 min)

Before we start analyzing data, let's set up version control. You will practice a standard GitHub workflow: **fork → clone → branch → commit → push → pull request**.

### Step 1 — Fork the repository

Go to the workshop repository on GitHub and click **Fork** (top-right). This creates your own copy.

### Step 2 — Clone your fork

```bash
git clone https://github.com/<YOUR-USERNAME>/Workshop-EDA.git
cd Workshop-EDA
```

### Step 3 — Create a working branch

```bash
git checkout -b eda-workshop
```

You will commit your progress to this branch as you work through the exercises.

---

## Setup

Run this cell to import libraries. If any are missing, install with `pip install <package>`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Nicer defaults
sns.set_theme(style="whitegrid", palette="mut"
                                         "ed")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 20)

print("All imports OK ✓")

All imports OK ✓


---
## Part 1: First Contact with the Data (~20 min)

Before doing anything fancy, get oriented. Think of yourself as a detective arriving at a scene — look around before touching anything.

### Exercise 1.1 ⭐ — Load and inspect

Load your dataset and answer these questions:
- How many rows and columns?
- What are the column names and data types?
- Show the first 5 and last 5 rows

Useful functions: `pd.read_csv()`, `.shape`, `.dtypes`, `.head()`, `.tail()`, `.info()`

In [4]:
# Your code here
# df = pd.read_csv("your_file.csv")              # ← your own dataset
# df = pd.read_csv("data/bike_sharing_eda.csv")   # ← or the provided dataset
df = pd.read_csv("data/bike_sharing_eda.csv")
df.shape
df.dtypes
df.head()
df.tail()
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 803 entries, 0 to 802
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 803 non-null    str    
 1   station              803 non-null    str    
 2   user_type            803 non-null    str    
 3   age                  730 non-null    float64
 4   trip_duration_min    803 non-null    float64
 5   temperature_c        803 non-null    float64
 6   humidity_pct         732 non-null    float64
 7   wind_speed_kmh       738 non-null    float64
 8   weather              803 non-null    str    
 9   daily_station_trips  803 non-null    int64  
 10  satisfaction_score   687 non-null    float64
dtypes: float64(6), int64(1), str(4)
memory usage: 69.1 KB


### Exercise 1.2 ⭐ — Describe the variables

Fill in a table like this (markdown or comments) for your dataset:

| Column | Type (numerical/categorical/date) | What does it represent? |
|--------|-----------------------------------|------------------------|
| ...    | ?                                 | ?                      |

- Which columns are numerical? Categorical? Dates? IDs?
- Are there columns you don't understand or that need documentation?

**Tip:** Use `.nunique()` and `.describe()` to help classify variables.

In [ ]:
# Your code here


### Exercise 1.3 ⭐⭐ — Data types check

Are the pandas dtypes appropriate for each column? Common issues:
- Date columns stored as strings instead of `datetime`
- Categorical columns stored as `object` instead of `category`
- Numeric columns read as strings (e.g., because of stray commas or units)

Fix any dtype issues you find.

### 🔀 Git checkpoint

Save your progress so far:

```bash
git add eda_workshop.ipynb
git commit -m "Complete Part 1: first contact with data"
```

In [ ]:
# Your code here


---
## Part 2: Data Quality — Cleaning & Consistency (~25 min)

Real data is never clean. Here we check for the most common problems.

### Exercise 2.1 ⭐ — Duplicates

- How many duplicate rows are there?
- Show the duplicated rows. Are they exact duplicates?
- Decide: should you drop them? Do it.

Useful: `.duplicated()`, `.drop_duplicates()`

In [ ]:
# Your code here


### Exercise 2.2 ⭐ — Inconsistent categories

Check the unique values in your categorical columns. Do you spot any inconsistencies?
- Mixed case ("Yes" vs "yes" vs "YES")?
- Leading/trailing whitespace?
- Typos or near-duplicates?

Clean them up.

Useful: `.unique()`, `.value_counts()`, `.str.lower()`, `.str.strip()`, `.replace()`

> 💡 *Bike-sharing dataset:* Check `weather`, `user_type`, and `station`.

In [ ]:
# Your code here


### Exercise 2.3 ⭐⭐ — Outliers and impossible values

Look at your numerical columns. Are there any values that seem impossible or clearly erroneous?

- What is the valid range for each variable? Are there values outside it?
- Are there zeros or negatives where they shouldn't exist?
- Are there extreme values that could be data entry errors?

Identify suspicious values. For now, just flag them — we'll decide what to do later.

Useful: `.describe()`, box plots, `.query()`

> 💡 *Bike-sharing dataset:* Look at `trip_duration_min` — can a trip last negative minutes? 1440 minutes (= 24 hours)?

In [ ]:
# Your code here


---
## Part 3: Missing Values Analysis (~30 min)

This is the heart of the workshop. Missing data is one of the most common and most consequential data quality issues. **How** data is missing matters as much as **how much** is missing.

### Types of missingness

| Type | Abbreviation | Meaning | Example |
|------|-------------|---------|--------|
| Missing Completely At Random | MCAR | Missingness is unrelated to any variable | Sensor randomly fails |
| Missing At Random | MAR | Missingness depends on *observed* variables | Casual users skip surveys more |
| Missing Not At Random | MNAR | Missingness depends on the *missing value itself* | Older people hide their age |

### Exercise 3.1 ⭐ — Missing values overview

- Which columns have missing values?
- How many and what percentage of values are missing in each?
- Create a bar chart showing the count of missing values per column.

Useful: `.isnull().sum()`, `.isnull().mean()`

In [ ]:
# Your code here


### Exercise 3.2 ⭐⭐ — Visualize missing data patterns

A simple table of counts doesn't tell you *where* or *how* values are missing. Visualize the missingness pattern.

**Option A** (simple): Create a heatmap where rows = observations, columns = variables, colored by missing/not missing.

**Option B** (recommended): Install and use the `missingno` library:
```python
pip install missingno
import missingno as msno
msno.matrix(df)
msno.heatmap(df)  # correlations between missing values
```

What patterns do you notice? Do some variables tend to be missing together?

In [ ]:
# Your code here


### Exercise 3.3 ⭐⭐ — Is any variable Missing At Random (MAR)?

Pick a column with missing values. Does the missingness depend on another (observed) column?

1. Group by a categorical variable and calculate the proportion of missing values in your target column
2. Visualize this with a bar chart
3. If the rates differ significantly across groups, this suggests MAR

*Think about it:* Why would one group be less likely to have this value recorded?

> 💡 *Bike-sharing dataset:* Check whether `satisfaction_score` missingness depends on `user_type`.

In [ ]:
# Your code here


### Exercise 3.4 ⭐⭐ — Another MAR investigation

Pick a different column with missing values. Can you find another observed variable that predicts its missingness?

1. For each group/category, what fraction of values are missing?
2. Visualize it.
3. Can you think of a real-world reason for this pattern?

> 💡 *Bike-sharing dataset:* Check whether `humidity_pct` missingness depends on `weather`. Why might humidity sensors fail in rain or snow?

In [ ]:
# Your code here


### Exercise 3.5 ⭐⭐⭐ — Could any variable be Missing Not At Random (MNAR)?

MNAR is the trickiest case — missingness depends on the missing value itself. We can't directly test this (we don't have the missing values!), but we can look for indirect evidence:

1. For a column with missing values, look at the distribution of the *observed* values. Does something look "chopped" or unusual?
2. Create a binary column `col_missing` (True/False) and check if it correlates with other variables that might proxy for the missing one.
3. Does the missing rate make sense given the domain?

> 💡 *Bike-sharing dataset:* Look at `age`. Is there a suspicious drop-off at higher ages? Older people might be less likely to report their age (MNAR).

In [ ]:
# Your code here


### 🔀 Git checkpoint

You're halfway through! Commit your work:

```bash
git add eda_workshop.ipynb
git commit -m "Complete Parts 2-3: data quality and missing values"
```

### Exercise 3.6 ⭐⭐⭐ — Impact of missing data on analysis

Missing data can bias your results. Demonstrate this with your dataset:

1. Pick a numeric column with missing values. Calculate its overall mean (dropping NaN).
2. Calculate the mean *by group* (using a relevant categorical column).
3. If some groups have more missing data, is the overall mean biased? In which direction?
4. *Optional:* What would the mean look like if you imputed missing values with the group mean instead of dropping them?

> 💡 *Bike-sharing dataset:* Casual users have ~30% missing `satisfaction_score` vs ~5% for subscribers. How does this bias the overall mean?

In [ ]:
# Your code here


---
## Part 4: Distributions & Visualization (~25 min)

Now that you understand the data quality, let's explore what the data *tells us*.

### Exercise 4.1 ⭐ — Univariate distributions

Plot histograms or KDE plots for your numerical columns.

For each, note:
- Is it roughly symmetric or skewed?
- Are there any obvious outliers?
- What is the typical (median) value?

Useful: `sns.histplot()`, `sns.kdeplot()`, `.describe()`

In [ ]:
# Your code here


### Exercise 4.2 ⭐ — Categorical distributions

Plot bar charts for your categorical columns.

- Are any categories dominant or very rare?
- Is the distribution balanced or heavily skewed?
- Are there categories that should be merged or renamed?

Useful: `sns.countplot()`, `.value_counts().plot.bar()`

In [ ]:
# Your code here


### Exercise 4.3 ⭐⭐ — Bivariate relationships

Explore relationships between pairs of variables in your dataset:

1. **Numerical vs. numerical:** Pick two numerical columns and create a scatter plot. Is there a relationship?
2. **Categorical vs. numerical:** Pick a categorical and a numerical column. Use a box plot to compare distributions across groups.
3. Do any relationships surprise you?

Useful: `sns.scatterplot()`, `sns.boxplot()`, `sns.violinplot()`

In [ ]:
# Your code here


### Exercise 4.4 ⭐⭐ — Correlation matrix

Compute and visualize a correlation matrix for all numerical columns.

- Which pairs are most correlated?
- Any surprising correlations or non-correlations?

Useful: `.corr()`, `sns.heatmap(annot=True)`

In [ ]:
# Your code here


### Exercise 4.5 ⭐⭐⭐ — Temporal or grouped patterns

If your data has a time dimension:
1. Parse dates as `datetime` if you haven't already
2. Plot a value over time. Add a rolling average to smooth it.
3. Are there seasonal, weekly, or other periodic patterns?

If no time dimension, try a grouped analysis:
- Use `groupby` to aggregate by a meaningful category and visualize trends across groups.

Useful: `.resample()`, `.rolling()`, `plt.twinx()` for dual y-axes

In [ ]:
# Your code here


---
## Part 5: Synthesis & Reporting (~15 min)

The most important part of EDA is *communicating what you found*.

### Exercise 5.1 ⭐ — Write your EDA summary

In a markdown cell below, write a short (5–10 sentences) summary of your findings. Structure it around:

1. **Dataset overview** — size, variables, time period
2. **Data quality issues found** — duplicates, inconsistencies, outliers
3. **Missing data** — which variables, how much, what type (MCAR/MAR/MNAR)
4. **Key patterns** — 2-3 interesting relationships or distributions
5. **Recommendations** — what would you do before modeling this data?

*Write your summary here*



### Exercise 5.2 ⭐⭐⭐ — The "one chart" challenge

Create a single, polished visualization that communicates the most interesting finding from your EDA. This should be a chart you could show to a non-technical stakeholder.

Requirements:
- Clear title
- Axis labels
- Legend if needed
- Annotation or caption explaining the insight

---
## Wrap-up: Push & Pull Request (~5 min)

You're done! Let's submit your work via GitHub.

### Step 1 — Final commit

```bash
git add eda_workshop.ipynb
git commit -m "Complete EDA workshop"
```

### Step 2 — Push your branch

```bash
git push -u origin eda-workshop
```

### Step 3 — Open a Pull Request

1. Go to your fork on GitHub
2. You should see a banner suggesting to create a pull request — click it
3. Set the base repository to the original workshop repo and base branch to `main`
4. Title your PR: **"EDA workshop — \<your name\>"**
5. In the description, paste your EDA summary from Exercise 5.1

Congratulations — you have practiced both EDA and a real-world Git workflow!

In [ ]:
# Your code here


---
## Bonus Exercises (if you finish early)

### Bonus A ⭐⭐⭐ — Grouped summary table
Use `groupby` on a meaningful categorical column to create a summary table with aggregated statistics (means, counts, missing percentages). Which group stands out?

### Bonus B ⭐⭐⭐ — Pairplot
Use `sns.pairplot()` colored by a categorical variable to explore all numerical relationships at once. What patterns jump out?

### Bonus C ⭐⭐⭐ — Missing data imputation comparison
For a column with missing values, compare three strategies:
1. Drop all rows with missing values
2. Fill with the overall mean
3. Fill with the group mean (by a relevant categorical variable)

How does each strategy affect the overall mean and the distribution?

In [ ]:
# Bonus exercises here
